# Course 4 - Applied Text Mining in Python
## Assignment 4: Document Similarity and Topic Modelling

### Overview
This assignment covers two advanced NLP techniques. Part 1 uses WordNet path similarity to measure semantic similarity between documents — a knowledge-based approach that leverages lexical ontology rather than raw word overlap. Part 2 applies Latent Dirichlet Allocation (LDA) to discover latent topics in a newsgroup dataset.

### Learning Objectives
- Convert raw text to WordNet synsets via POS tagging
- Compute normalized path similarity between synset lists
- Use WordNet-based similarity to classify paraphrase pairs
- Train an LDA model with Gensim on a text corpus
- Extract topic distributions and assign human-readable topic names

### Datasets
- **Part 1:** `assets/paraphrases.csv` — 20 document pairs with binary paraphrase labels
- **Part 2:** `assets/newsgroups` — newsgroup posts (pickled), used to train LDA with 10 topics

### Assignment Structure

**Part 1 — Document Path Similarity**
| Function | Task |
|----------|------|
| `doc_to_synsets(doc)` | Tokenize, POS-tag, and convert words to first-match WordNet synsets |
| `similarity_score(s1, s2)` | For each synset in s1, find max path similarity to any synset in s2; normalize |
| `most_similar_docs()` | Find the paraphrase pair with highest symmetrical similarity score |
| `label_accuracy()` | Label pairs as paraphrase if score > 0.75; report sklearn accuracy |

**Part 2 — Topic Modelling with LDA**
| Function | Task |
|----------|------|
| `ldamodel` | Train LDA with 10 topics, passes=25, random_state=34 |
| `lda_topics()` | Return top 10 words per topic |
| `topic_distribution()` | Get topic distribution for a new astronomy document |
| `topic_names()` | Assign human-readable names to the 10 LDA topics |

### Key Concepts
- **Path Similarity:** Measures semantic relatedness via shortest path in WordNet hierarchy (0=unrelated, 1=identical)
- **LDA:** Probabilistic model that assumes each document is a mixture of topics, each topic a mixture of words
- **Gensim Corpus:** `gensim.matutils.Sparse2Corpus` converts sklearn sparse matrices to Gensim's format

## Part 1 - Document Similarity

For the first part of this assignment, you will complete the functions `doc_to_synsets` and `similarity_score` which will be used by `document_path_similarity` to find the path similarity between two documents.

The following functions are provided:
* **`convert_tag:`** converts the tag given by `nltk.pos_tag` to a tag used by `wordnet.synsets`. You will need to use this function in `doc_to_synsets`.
* **`document_path_similarity:`** computes the symmetrical path similarity between two documents by finding the synsets in each document using `doc_to_synsets`, then computing similarities using `similarity_score`.

You will need to finish writing the following functions:
* **`doc_to_synsets:`** returns a list of synsets in document. This function should first tokenize and part of speech tag the document using `nltk.word_tokenize` and `nltk.pos_tag`. Then it should find each tokens corresponding synset using `wn.synsets(token, wordnet_tag)`. The first synset match should be used. If there is no match, that token is skipped.
* **`similarity_score:`** returns the normalized similarity score of a list of synsets (s1) onto a second list of synsets (s2). For each synset in s1, find the synset in s2 with the largest similarity value. Sum all of the largest similarity values together and normalize this value by dividing it by the number of largest similarity values found. Be careful with data types, which should be floats. Missing values should be ignored.

Once doc_to_synsets and similarity_score have been completed, submit to the autograder which will run a test to check that these functions are running correctly.

*Do not modify the functions `convert_tag` and `document_path_similarity`.*

In [1]:
%%capture
import numpy as np
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.corpus import wordnet as wn
from icecream import ic
import pandas as pd
nltk.data.path.append("assets/")

def convert_tag(tag):
    """Convert the tag given by nltk.pos_tag to the tag used by wordnet.synsets"""
    
    tag_dict = {'N': 'n', 'J': 'a', 'R': 'r', 'V': 'v'}
    try:
        return tag_dict[tag[0]]
    except KeyError:
        return None

In [2]:



def doc_to_synsets(doc):
    """
    Returns a list of synsets in document.

    Tokenizes and tags the words in the document doc.
    Then finds the first synset for each word/tag combination.
    If a synset is not found for that combination it is skipped.

    Args:
        doc: string to be converted

    Returns:
        list of synsets

    Example:
        doc_to_synsets('Fish are friends.')
        Out: [Synset('fish.n.01'), Synset('be.v.01'), Synset('friend.n.01')]
    """

    # YOUR CODE HERE
    from icecream import ic
    from nltk.corpus import wordnet
    from nltk.tokenize import word_tokenize
    import nltk

    #document_str = "Today is going be a bright sunny day"
    document_str=doc
    #ic(document_str)
    token_lst = word_tokenize(document_str)
    #ic(token_lst)

    tagged_token_lst = nltk.pos_tag(token_lst)
    #ic(tagged_token_lst)    

    #ic(tagged_token_lst)
    list_of_synsets = []
    wordnet_tag_list =[]
    for token in tagged_token_lst:
        wordnet_tag = convert_tag(token[1])

        #ic(token[1],wordnet_tag)
        wordnet_tag_list.append(wordnet_tag)
        synsets_lst = wordnet.synsets(token[0], pos = wordnet_tag)
        #ic(synsets_lst)
        #ic(type(synsets_lst))
        #if syn !=None: 
            #syn_01 = syn[0]
        #ic(len(synsets))
        #ic(type(syn))
        #ic(len)
        #ic(synsets_lst[0])
        #ic(len(synsets_lst))
        
        if len(synsets_lst)== 0 :
            #ic(token[0], wordnet_tag, synsets[0])
            continue
        else:
            list_of_synsets.append(synsets_lst[0])
        #else:
            #ic(token[0], wordnet_tag, None)
        #    list_of_synsets.append(None)        
    #ic(wordnet_tag_list)
    #ic(list_of_synsets)

    #raise NotImplementedError()
    return list_of_synsets # Your Answer Here


def similarity_score(s1, s2):
    """
    Calculate the normalized similarity score of s1 onto s2

    For each synset in s1, finds the synset in s2 with the largest similarity value.
    Sum of all of the largest similarity values and normalize this value by dividing it by the
    number of largest similarity values found.

    Args:
        s1, s2: list of synsets from doc_to_synsets

    Returns:
        normalized similarity score of s1 onto s2

    Example:
        synsets1 = doc_to_synsets('I like cats')
        synsets2 = doc_to_synsets('I like dogs')
        similarity_score(synsets1, synsets2)
        Out: 0.7333333333333333
    """

    # YOUR CODE HERE
    import nltk
    nltk.download('wordnet_ic')
    from nltk.corpus import wordnet_ic
    

    brown_ic = wordnet_ic.ic('ic-brown.dat')

    #sent1 = "Today is Friday"
    #sent2 = "How is your day going?"
    synset1_lst = s1
    synset2_lst = s2

    #synset1_lst = doc_to_synsets(sent1)
    #synset2_lst = doc_to_synsets(sent2)
    max_score_lst =[]
    for synset1 in synset1_lst:
        curr_score = 0 
        for synset2 in synset2_lst:    
            #ic(synset1)
            #ic(synset2)
            score = synset1.path_similarity(synset2)
            if score > curr_score:
                curr_score =score
            #ic(score)    
        max_score_lst.append(curr_score)
    #ic(max_score_lst)
    normalized_score = sum(max_score_lst)/len(max_score_lst)
    #ic(normalized_score)

    #raise NotImplementedError()
    return normalized_score# Your Answer Here

In [3]:
def document_path_similarity(doc1, doc2):
    """Finds the symmetrical similarity between doc1 and doc2"""

    synsets1 = doc_to_synsets(doc1)
    synsets2 = doc_to_synsets(doc2)

    #ic(synsets1, synsets2)
    
    return (similarity_score(synsets1, synsets2) + similarity_score(synsets2, synsets1)) / 2

`paraphrases` is a DataFrame which contains the following columns: `Quality`, `D1`, and `D2`.

`Quality` is an indicator variable which indicates if the two documents `D1` and `D2` are paraphrases of one another (1 for paraphrase, 0 for not paraphrase).

In [4]:
# Use this dataframe for questions most_similar_docs and label_accuracy
paraphrases = pd.read_csv('assets/paraphrases.csv')
paraphrases.head()

,Quality,D1,D2
0,1,"Ms Stewart, the chief executive, was not expec...","Ms Stewart, 61, its chief executive officer an..."
1,1,After more than two years' detention under the...,After more than two years in detention by the ...
2,1,"""It still remains to be seen whether the reven...","""It remains to be seen whether the revenue rec..."
3,0,"And it's going to be a wild ride,"" said Allan ...","Now the rest is just mechanical,"" said Allan H..."
4,1,The cards are issued by Mexico's consulates to...,The card is issued by Mexico's consulates to i...


___

### most_similar_docs

Using `document_path_similarity`, find the pair of documents in paraphrases which has the maximum similarity score.

*This function should return a tuple `(D1, D2, similarity_score)`*

In [5]:
def most_similar_docs():
    
     # YOUR CODE HERE

    #ic(paraphrases.columns)

    paraphrases['D1'] =paraphrases['D1'].astype('str')
    paraphrases['D2'] =paraphrases['D2'].astype('str')
    #ic(paraphrases.shape)
    
    #for index,row in paraphrases.iterrows():

    row_count, col_count = paraphrases.shape
    #ic(range(row_count))

    sim_score =[]
    for row_count in range(row_count):

        #if row_count ==1:
        #    break
        
        #ic(index,row)
        #ic(type(row))
        #ic(type(row['D1']))
        
        doc1 = paraphrases['D1'][row_count]
        doc2 = paraphrases['D2'][row_count]

        
        #ic(doc1)
        #ic(doc2)
        score_d1_d2 = document_path_similarity(doc1, doc2)
        #ic(score_d1_d2)
        sim_score.append((row_count, score_d1_d2))
        
        
    #ic(sim_score)
    score_lst = [score[1] for score in sim_score]
    max_score = max(score_lst)
    #index_of_max = list.index(max_score)
    ic(sim_score)
    #ic(score_lst)
    ic(max_score)
    #ic(list)
    index_of_max = score_lst.index(max_score)

    return (paraphrases['D1'][index_of_max], paraphrases['D2'][index_of_max], max_score)
    #return 
    #raise NotImplementedError()
    #return # Your Answer Here

most_similar_docs()

ic

|

sim_score

:

[

...

|

max_score

:

0.9590643274853801

('"Indeed, Iran should be put on notice that efforts to try to remake Iraq in their image will be aggressively put down," he said.',
 '"Iran should be on notice that attempts to remake Iraq in Iran\'s image will be aggressively put down," he said.\n',
 0.9590643274853801)

### label_accuracy

Provide labels for the twenty pairs of documents by computing the similarity for each pair using `document_path_similarity`. Let the classifier rule be that if the score is greater than 0.75, label is paraphrase (1), else label is paraphrase (0). Report accuracy of the classifier using scikit-learn's accuracy_score.

*This function should return a float.*

In [6]:
def label_accuracy():
    from sklearn.metrics import accuracy_score

    # YOUR CODE HERE
    paraphrases['D1'] =paraphrases['D1'].astype('str')
    paraphrases['D2'] =paraphrases['D2'].astype('str')
    #ic(paraphrases.shape)
    #for index,row in paraphrases.iterrows():

    row_count, col_count = paraphrases.shape
    #ic(range(row_count))

    sim_score =[]
    for row_count in range(row_count):

        #if row_count ==1:
        #    break
        
        #ic(index,row)
        #ic(type(row))
        #ic(type(row['D1']))
        
        doc1 = paraphrases['D1'][row_count]
        doc2 = paraphrases['D2'][row_count]

        
        #ic(doc1)
        #ic(doc2)
        score_d1_d2 = document_path_similarity(doc1, doc2)
        #ic(score_d1_d2)
        sim_score.append((row_count, score_d1_d2))
        
        
    #ic(sim_score)
    score_lst = [score[1] for score in sim_score]
    max_score = max(score_lst)
    #index_of_max = list.index(max_score)
    #ic(score_lst)
    #ic(max_score)
    #ic(list)
    #index_of_max = score_lst.index(max_score)
    paraphrases['label']=0

    lst_label =[]
    for row_count, score in sim_score: 
        if score > 0.75:
            #paraphrases['label'][row_count]=1
            lst_label.append(1)
        else:
            lst_label.append(0)

    #ic(lst_label)
    combined_lst = list(zip(lst_label, paraphrases['Quality'].values))
    ic(combined_lst)
    #return (paraphrases['D1'][index_of_max], paraphrases['D2'][index_of_max], max_score)
    quality_lbl = paraphrases['Quality'].values

    score = accuracy_score(lst_label,quality_lbl )
    #ic(score)
    return score
    
    raise NotImplementedError()
    
    return # Your Answer Here

label_accuracy()

ic

|

combined_lst

:

[

...

,

1

)

]

0.7

## Test similarity_score

In [7]:
doc1="I don't like green eggs and ham"
doc2="Bacon and eggs with a dose of Iodine"
synsets1 = doc_to_synsets(doc1)
#display(synsets1)
synsets2 = doc_to_synsets(doc2)
#display(synsets2)

print(similarity_score(synsets1, synsets2))
print(similarity_score(synsets2, synsets1))
print(document_path_similarity(doc1, doc2))  

0.4573412698412698
0.5116666666666666


0.48450396825396824


## Part 2 - Topic Modelling

For the second part of this assignment, you will use Gensim's LDA (Latent Dirichlet Allocation) model to model topics in `newsgroup_data`. You will first need to finish the code in the cell below by using gensim.models.ldamodel.LdaModel constructor to estimate LDA model parameters on the corpus, and save to the variable `ldamodel`. Extract 10 topics using `corpus` and `id_map`, and with `passes=25` and `random_state=34`.

In [8]:
#LDA
#Choose length of document d
#Choose mix of topics for document d
#Use a topics multinomial distribution to output words to fill that topics quota


import pickle
import gensim
from sklearn.feature_extraction.text import CountVectorizer
from gensim import corpora, models 
import numpy as np 

# Load the list of documents
with open('assets/newsgroups', 'rb') as f:
    newsgroup_data = pickle.load(f)



# Use CountVectorizor to find three letter tokens, remove stop_words, 
# remove tokens that don't appear in at least 20 documents,
# remove tokens that appear in more than 20% of the documents
vect = CountVectorizer(min_df=20, max_df=0.2, stop_words='english', 
                       token_pattern='(?u)\\b\\w\\w\\w+\\b')

#ic(newsgroup_data[0:100])

# Fit and transform
X = vect.fit_transform(newsgroup_data)

feature_name_lst = vect.get_feature_names_out()

#feat_arr = np.array(feature_name_lst)
#ic(feat_arr[0:10])

# Convert sparse matrix to gensim corpus.
corpus = gensim.matutils.Sparse2Corpus(X, documents_columns=False)
#for doc in corpus:
#    ic(doc)
ic(vect.vocabulary_.items())

# Mapping from word IDs to words (To be used in LdaModel's id2word parameter)
id_map = dict((v, k) for k, v in vect.vocabulary_.items())


ic

|

vect

.

vocabulary_

...

,

659

)

]

)

### Approach: LDA Corpus Preparation
This cell prepares the text corpus for LDA topic modelling:
1. **CountVectorizer** tokenizes the newsgroup posts using a 3+ character word pattern, removes English stop words, and filters very rare (< 20 docs) and very common (> 20% of docs) terms
2. `Sparse2Corpus` converts the sklearn sparse matrix to Gensim's corpus format (list of (word_id, count) pairs per document)
3. `id_map` creates a word ID to word string mapping needed by LdaModel's `id2word` parameter

LDA assumes each document is a mixture of topics and each topic is a mixture of words. `passes=25` means the model iterates through the entire corpus 25 times during training.

In [9]:

# Use the gensim.models.ldamodel.LdaModel constructor to estimate 
# LDA model parameters on the corpus, and save to the variable `ldamodel`

ldamodel = None
# YOUR CODE HERE
ldamodel = gensim.models.ldamodel.LdaModel(corpus, num_topics =10, id2word = id_map, passes = 25, random_state =34  )


#raise NotImplementedError()

### lda_topics

Using `ldamodel`, find a list of the 10 topics and the most significant 10 words in each topic. This should be structured as a list of 10 tuples where each tuple takes on the form:

`(9, '0.068*"space" + 0.036*"nasa" + 0.021*"science" + 0.020*"edu" + 0.019*"data" + 0.017*"shuttle" + 0.015*"launch" + 0.015*"available" + 0.014*"center" + 0.013*"information"')`

for example.

*This function should return a list of tuples.*

In [10]:
def lda_topics():
    
    # YOUR CODE HERE

    topics = ldamodel.print_topics(num_topics =10, num_words =10)
    return topics
    #raise NotImplementedError()
    #return # Your Answer Here
lda_topics()    

[(0,
  '0.056*"edu" + 0.043*"com" + 0.033*"thanks" + 0.022*"mail" + 0.021*"know" + 0.020*"does" + 0.014*"info" + 0.012*"monitor" + 0.010*"looking" + 0.010*"don"'),
 (1,
  '0.024*"ground" + 0.018*"current" + 0.018*"just" + 0.013*"want" + 0.013*"use" + 0.011*"using" + 0.011*"used" + 0.010*"power" + 0.010*"speed" + 0.010*"output"'),
 (2,
  '0.061*"drive" + 0.042*"disk" + 0.033*"scsi" + 0.030*"drives" + 0.028*"hard" + 0.028*"controller" + 0.027*"card" + 0.020*"rom" + 0.018*"floppy" + 0.017*"bus"'),
 (3,
  '0.023*"time" + 0.015*"atheism" + 0.014*"list" + 0.013*"left" + 0.012*"alt" + 0.012*"faq" + 0.012*"probably" + 0.011*"know" + 0.011*"send" + 0.010*"months"'),
 (4,
  '0.025*"car" + 0.016*"just" + 0.014*"don" + 0.014*"bike" + 0.012*"good" + 0.011*"new" + 0.011*"think" + 0.010*"year" + 0.010*"cars" + 0.010*"time"'),
 (5,
  '0.030*"game" + 0.027*"team" + 0.023*"year" + 0.017*"games" + 0.016*"play" + 0.012*"season" + 0.012*"players" + 0.012*"win" + 0.011*"hockey" + 0.011*"good"'),
 (6,
  '0.0

### topic_distribution

For the new document `new_doc`, find the topic distribution. Remember to use vect.transform on the the new doc, and Sparse2Corpus to convert the sparse matrix to gensim corpus.

*This function should return a list of tuples, where each tuple is `(#topic, probability)`*

In [11]:
new_doc = ["\n\nIt's my understanding that the freezing will start to occur because \
of the\ngrowing distance of Pluto and Charon from the Sun, due to it's\nelliptical orbit. \
It is not due to shadowing effects. \n\n\nPluto can shadow Charon, and vice-versa.\n\nGeorge \
Krumins\n-- "]

In [12]:
def topic_distribution():
    """
    Transform new_doc using the already-fitted vect, convert to Gensim corpus,
    then query the trained ldamodel for topic probabilities.
    Returns a list of (topic_id, probability) tuples.
    """
    import gensim

    # Transform new document using the pre-fitted CountVectorizer
    X_new = vect.transform(new_doc)

    # Convert sparse sklearn matrix to Gensim corpus format
    corpus_new = gensim.matutils.Sparse2Corpus(X_new, documents_columns=False)

    # Get topic distribution for the single document
    topic_dist = list(ldamodel[corpus_new])[0]
    return topic_dist

topic_distribution()

[(0, 0.020003108),
 (1, 0.020003324),
 (2, 0.020001281),
 (3, 0.49674895),
 (4, 0.020004038),
 (5, 0.020004129),
 (6, 0.020002972),
 (7, 0.020002645),
 (8, 0.020003129),
 (9, 0.34322643)]

### topic_names

From the list of the following given topics, assign topic names to the topics you found. If none of these names best matches the topics you found, create a new 1-3 word "title" for the topic.

Topics: Health, Science, Automobiles, Politics, Government, Travel, Computers & IT, Sports, Business, Society & Lifestyle, Religion, Education.

*This function should return a list of 10 strings.*

In [13]:
def topic_names():
    
    # YOUR CODE HERE
    #topic_dis = ldamodel(new_doc)
    #print(topic_dis[0])
    display(ldamodel.print_topics(num_topics =10, num_words =10))

    #raise NotImplementedError()
    return ['Education', 
            'Business',
            'Computers & IT',
             'Religion',
             'Automobiles',
             'Sports',
             'Health',
             'Society & Lifestyle',
             'Computers & IT',
             'Science'] # Your Answer Here
topic_names()

[(0,
  '0.056*"edu" + 0.043*"com" + 0.033*"thanks" + 0.022*"mail" + 0.021*"know" + 0.020*"does" + 0.014*"info" + 0.012*"monitor" + 0.010*"looking" + 0.010*"don"'),
 (1,
  '0.024*"ground" + 0.018*"current" + 0.018*"just" + 0.013*"want" + 0.013*"use" + 0.011*"using" + 0.011*"used" + 0.010*"power" + 0.010*"speed" + 0.010*"output"'),
 (2,
  '0.061*"drive" + 0.042*"disk" + 0.033*"scsi" + 0.030*"drives" + 0.028*"hard" + 0.028*"controller" + 0.027*"card" + 0.020*"rom" + 0.018*"floppy" + 0.017*"bus"'),
 (3,
  '0.023*"time" + 0.015*"atheism" + 0.014*"list" + 0.013*"left" + 0.012*"alt" + 0.012*"faq" + 0.012*"probably" + 0.011*"know" + 0.011*"send" + 0.010*"months"'),
 (4,
  '0.025*"car" + 0.016*"just" + 0.014*"don" + 0.014*"bike" + 0.012*"good" + 0.011*"new" + 0.011*"think" + 0.010*"year" + 0.010*"cars" + 0.010*"time"'),
 (5,
  '0.030*"game" + 0.027*"team" + 0.023*"year" + 0.017*"games" + 0.016*"play" + 0.012*"season" + 0.012*"players" + 0.012*"win" + 0.011*"hockey" + 0.011*"good"'),
 (6,
  '0.0

['Education',
 'Business',
 'Computers & IT',
 'Religion',
 'Automobiles',
 'Sports',
 'Health',
 'Society & Lifestyle',
 'Computers & IT',
 'Science']